# Estimator-based VQE energy scan

Evaluate a compact two-qubit chemistry-style Hamiltonian over a variational ansatz parameter scan.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [2]:
hamiltonian = SparsePauliOp.from_list([
    ("II", -1.05), ("ZI", 0.39), ("IZ", -0.39), ("ZZ", -0.01), ("XX", 0.18)
])
angles = np.linspace(-np.pi, np.pi, 25)
circuits = []
for angle in angles:
    circuit = QuantumCircuit(2)
    circuit.ry(float(angle), 0)
    circuit.cx(0, 1)
    circuit.ry(float(-0.37 * angle), 1)
    circuits.append(circuit)

def reference_energies():
    estimator = StatevectorEstimator()
    return np.asarray([estimator.run([(c, hamiltonian)]).result()[0].data.evs.item() for c in circuits])

reference, reference_ms, _ = benchmark(reference_energies)
backend = MettleQBackend(method="statevector", device="cpu")
compiled = [transpile(c, backend, optimization_level=1) for c in circuits]
estimator = MettleQEstimatorV2(backend=backend)

def mettleq_energies():
    return np.asarray([estimator.run([(c, hamiltonian)]).result()[0].data.evs.item() for c in compiled])

candidate, mettleq_ms, _ = benchmark(mettleq_energies)
error = max_abs_error(reference, candidate)
minima_match = int(np.argmin(reference)) == int(np.argmin(candidate))
method, device = qiskit_selection(estimator)
tutorial_result = emit_result(
    notebook="qiskit/10_vqe.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="VQE energy trace atol=3e-6",
    passed=error <= 3e-6 and minima_match,
    exact_match=minima_match,
    selected_method=method,
    selected_device=device,
    metrics={"max_energy_error": error, "reference_minimum": float(reference.min()), "mettleq_minimum": float(candidate.min()), "minimum_index": int(np.argmin(candidate))},
)

TUTORIAL_RESULT::{"check": "VQE energy trace atol=3e-6", "exact_match": true, "framework": "qiskit", "machine": "arm64", "metrics": {"max_energy_error": 1.3451273783715578e-07, "mettleq_minimum": -1.2243290054798128, "minimum_index": 7, "reference_minimum": -1.2243289931098607}, "mettleq_median_ms": 81.27641698229127, "notebook": "qiskit/10_vqe.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 7.833499985281378, "reference_over_mettleq": 0.09638097096465464, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}
